In [12]:
%pip install selenium webdriver-manager beautifulsoup4

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import csv
import time

def scrape_olx():
    """
    Skrypt do pobierania danych o ofertach mieszkań na sprzedaż z serwisu OLX.pl
    używając Selenium do kontroli przeglądarki i BeautifulSoup do parsowania HTML.
    Dane zapisywane są do pliku olx_offers.csv.
    """
    print("Inicjalizacja przeglądarki Chrome...")
    
    # Dodanie opcji Chrome dla lepszej stabilności
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    # Ukrycie automatyzacji
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    start_url = "https://www.olx.pl/nieruchomosci/mieszkania/sprzedaz/"
    driver.get(start_url)
    
    try:
        print("Oczekiwanie na przycisk akceptacji cookies...")
        wait = WebDriverWait(driver, 10)
        cookies_button = wait.until(EC.element_to_be_clickable((By.ID, 'onetrust-accept-btn-handler')))
        cookies_button.click()
        print("Zaakceptowano cookies.")
    except TimeoutException:
        print("Nie znaleziono przycisku cookies lub upłynął limit czasu.")

    all_offers = []
    page_count = 1
    max_pages = 5  # Ograniczenie do 5 stron na potrzeby tego testu

    while page_count <= max_pages:
        print(f"Przetwarzanie strony: {page_count}")
        time.sleep(3)

        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')

        offers_list = soup.find_all('div', {'data-cy': 'l-card'})

        if not offers_list:
            print("Nie znaleziono ofert na stronie. Prawdopodobnie zmiana w strukturze strony.")
            break

        for offer_div in offers_list:
            try:
                # Różne możliwe selektory dla tytułu
                title_tag = (offer_div.find('h6') or 
                           offer_div.find('h4') or 
                           offer_div.find('a', class_='css-rc5s2u') or
                           offer_div.find('h6', class_='css-16v5mdi') or
                           offer_div.find('[data-cy="listing-ad-title"]'))
                
                price_tag = offer_div.find('p', {'data-testid': 'ad-price'})
                location_tag = offer_div.find('p', {'data-testid': 'location-date'})
                
                title = title_tag.text.strip() if title_tag else 'Brak tytułu'
                price = price_tag.text.strip() if price_tag else 'Brak ceny'
                location = location_tag.text.strip() if location_tag else 'Brak lokalizacji'
                
                # Pobieranie parametrów mieszkania (pokoje, metraż)
                # Szukamy w różnych możliwych miejscach
                area = 'Brak danych'
                rooms = 'Brak danych'
                
                # Próba 1: standardowe parametry
                attributes_list = offer_div.find_all('span', class_='css-643j0o')
                if attributes_list:
                    for attr in attributes_list:
                        attr_text = attr.text.strip()
                        if 'm²' in attr_text or 'metr' in attr_text.lower():
                            area = attr_text
                        elif any(word in attr_text.lower() for word in ['pokój', 'pokoje', 'pok']):
                            rooms = attr_text
                
                # Próba 2: inne możliwe selektory dla parametrów
                if area == 'Brak danych' or rooms == 'Brak danych':
                    param_elements = offer_div.find_all('span', class_='css-xl6fe0-Text')
                    for param in param_elements:
                        param_text = param.text.strip()
                        if 'm²' in param_text and area == 'Brak danych':
                            area = param_text
                        elif any(word in param_text.lower() for word in ['pokój', 'pokoje', 'pok']) and rooms == 'Brak danych':
                            rooms = param_text
                
                # Próba 3: szukanie w całym tekście oferty
                if area == 'Brak danych' or rooms == 'Brak danych':
                    all_text = offer_div.get_text()
                    import re
                    
                    if area == 'Brak danych':
                        area_match = re.search(r'(\d+[\.,]?\d*)\s*m²', all_text)
                        if area_match:
                            area = area_match.group(0)
                    
                    if rooms == 'Brak danych':
                        rooms_match = re.search(r'(\d+)\s*pok[ój|oje|oi]?', all_text.lower())
                        if rooms_match:
                            rooms = f"{rooms_match.group(1)} pokoje"

                offer_data = {
                    'title': title,
                    'price': price,
                    'location': location,
                    'area': area,
                    'rooms': rooms
                }
                all_offers.append(offer_data)
                
                # Debug info dla pierwszych kilku ofert
                if len(all_offers) <= 3:
                    print(f"Oferta {len(all_offers)}: {title[:50]}... | {price} | {rooms} | {area}")

            except Exception as e:
                print(f"Błąd podczas przetwarzania oferty: {e}")
                continue
        
        # Próba przejścia do następnej strony z lepszą obsługą błędów
        try:
            # Poczekaj na załadowanie elementów paginacji
            wait = WebDriverWait(driver, 10)
            
            # Spróbuj znaleźć przycisk następnej strony
            next_page_button = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[data-cy="pagination-forward"]')))
            
            # Sprawdź czy przycisk jest aktywny (nie ma klasy disabled)
            if "disabled" in next_page_button.get_attribute("class") or "":
                print("Przycisk 'Następna strona' jest nieaktywny. Zakończono.")
                break
            
            # Przewiń do przycisku aby był widoczny
            driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", next_page_button)
            time.sleep(2)
            
            # Poczekaj aż przycisk będzie klikalny
            next_page_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a[data-cy="pagination-forward"]')))
            
            # Spróbuj kliknąć standardowo
            try:
                next_page_button.click()
            except (ElementClickInterceptedException, StaleElementReferenceException):
                # Jeśli standardowy klik nie działa, użyj JavaScript
                driver.execute_script("arguments[0].click();", next_page_button)
            
            page_count += 1
            print(f"Przeszedłem do strony {page_count}")
            
        except TimeoutException:
            print("Nie znaleziono przycisku 'Następna strona' w określonym czasie. Koniec paginacji.")
            break
        except NoSuchElementException:
            print("Nie znaleziono przycisku 'Następna strona'. Koniec paginacji.")
            break
        except Exception as e:
            print(f"Błąd podczas przechodzenia do następnej strony: {e}")
            break
            
    driver.quit()
    print(f"Pobrano łącznie {len(all_offers)} ofert.")
    return all_offers

def save_to_csv(data, filename):
    """
    Zapisuje listę słowników do pliku CSV.
    """
    if not data:
        print("Brak danych do zapisania.")
        return
    
    fieldnames = data[0].keys() if data else []
    
    try:
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
        print(f"Dane zostały pomyślnie zapisane do pliku {filename}")
    except IOError as e:
        print(f"Błąd zapisu do pliku {filename}: {e}")

if __name__ == '__main__':
    olx_data = scrape_olx()
    if olx_data:
        save_to_csv(olx_data, 'olx_offers.csv')


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Inicjalizacja przeglądarki Chrome...
Oczekiwanie na przycisk akceptacji cookies...
Oczekiwanie na przycisk akceptacji cookies...
Zaakceptowano cookies.
Przetwarzanie strony: 1
Zaakceptowano cookies.
Przetwarzanie strony: 1
Oferta 1: AZOT CONCEPT: Nowe mieszkanie z windą i balkonem w... | 479 000 zł | Brak danych | 202563 m²
Oferta 2: GOTOWE ! Czeka na swojego właściciela!... | 493 500 zł | Brak danych | 202565,80 m²
Oferta 3: Apartament 59 m2 - LEVITYN , atrakcyjna lokalizacj... | 530 000 zł | Brak danych | 202559,60 m²
Oferta 1: AZOT CONCEPT: Nowe mieszkanie z windą i balkonem w... | 479 000 zł | Brak danych | 202563 m²
Oferta 2: GOTOWE ! Czeka na swojego właściciela!... | 493 500 zł | Brak danych | 202565,80 m²
Oferta 3: Apartament 59 m2 - LEVITYN , atrakcyjna lokalizacj... | 530 000 zł | Brak danych | 202559,60 m²
Przeszedłem do strony 2
Przetwarzanie strony: 2
Przeszedłem do strony 2
Przetwarzanie strony: 2
Przeszedł